#### Define notebook Parameters

Pass through th pl_worker

In [ ]:
# Framework-compatible parameters with manual fallbacks.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load taxonomy to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
prev_wm = None
curr_wm = None

# Source config
source_settings = json.dumps(
    {
        "source_name": "taxonomy",
        "source_connection": "shortcut",
        "source_schema": "external_curated_taxonomy",
        "table_config": [
            {
                "source_table": "curated_band_level",
                "target_table": "stage_taxonomy_curated_band_level",
                "target_pk": ["band_level_code"],
                "is_incremental": 0,
                "incremental_column": ""
            },
            {
                "source_table": "curated_bill_type",
                "target_table": "stage_taxonomy_curated_bill_type",
                "target_pk": ["bill_type_code"]
            },
            {
                "source_table": "curated_client_sector",
                "target_table": "stage_taxonomy_curated_client_sector",
                "target_pk": ["sic_code"]
            },
            {
                "source_table": "curated_company",
                "target_table": "stage_taxonomy_curated_company",
                "target_pk": ["company_code"]
            },
            {
                "source_table": "curated_country",
                "target_table": "stage_taxonomy_curated_country",
                "target_pk": ["country_code_iso2"]
            },
            {
                "source_table": "curated_currency",
                "target_table": "stage_taxonomy_curated_currency",
                "target_pk": ["currency_code"]
            },
            {
                "source_table": "curated_employee_sub_group",
                "target_table": "stage_taxonomy_curated_employee_sub_group",
                "target_pk": ["employee_sub_group_code"]
            },
            {
                "source_table": "curated_entry_classification",
                "target_table": "stage_taxonomy_curated_entry_classification",
                "target_pk": ["entry_classification_code"]
            },
            {
                "source_table": "curated_matter_billing_frequency",
                "target_table": "stage_taxonomy_curated_matter_billing_frequency",
                "target_pk": ["matter_billing_frequency_code"]
            },
            {
                "source_table": "curated_matter_billing_method",
                "target_table": "stage_taxonomy_curated_matter_billing_method",
                "target_pk": ["matter_billing_method_code"]
            },
            {
                "source_table": "curated_matter_category",
                "target_table": "stage_taxonomy_curated_matter_category",
                "target_pk": ["matter_category_code"]
            },
            {
                "source_table": "curated_matter_entry_type",
                "target_table": "stage_taxonomy_curated_matter_entry_type",
                "target_pk": ["matter_entry_type_code"]
            },
            {
                "source_table": "curated_matter_sector",
                "target_table": "stage_taxonomy_curated_matter_sector",
                "target_pk": ["sic_code"]
            },
            {
                "source_table": "curated_matter_status",
                "target_table": "stage_taxonomy_curated_matter_status",
                "target_pk": ["matter_status_code"]
            },
            {
                "source_table": "curated_nature_of_proceeding",
                "target_table": "stage_taxonomy_curated_nature_of_proceeding",
                "target_pk": ["nature_of_proceeding_code"]
            },
            {
                "source_table": "curated_office",
                "target_table": "stage_taxonomy_curated_office",
                "target_pk": ["office_code"]
            },
            {
                "source_table": "curated_partner_function",
                "target_table": "stage_taxonomy_curated_partner_function",
                "target_pk": ["partner_function_code"]
            },
            {
                "source_table": "curated_practice_group",
                "target_table": "stage_taxonomy_curated_practice_group",
                "target_pk": ["practice_group_code"]
            },
            {
                "source_table": "curated_team",
                "target_table": "stage_taxonomy_curated_team",
                "target_pk": ["team_code", "member_firm_code"]
            },
            {
                "source_table": "curated_transaction_document_type",
                "target_table": "stage_taxonomy_curated_transaction_document_type",
                "target_pk": ["transaction_document_type_code"]
            },
            {
                "source_table": "curated_work_type",
                "target_table": "stage_taxonomy_curated_work_type",
                "target_pk": ["technical_area_code"]
            }
        ]
    }
)

# Target config
target_settings = json.dumps(
    {
        "target_lakehouse": "lh_canada_global_mds",
        "target_schema": "stage_taxonomy",
        "target_location": "Files/stage/taxonomy",
        "target_load_strategy": "overwrite"
    }
)

#### Define functions and logging

Used for import functions

In [ ]:
%run nb_transformation_shared_functions

In [ ]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadTaxonomyToStage")
logger.setLevel(logging.INFO)

#### Define variables or parameters

Can manually run or be through worker

In [ ]:
# Accept either JSON strings from pipeline or already-parsed dicts for manual runs.
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings

# Source configuration
source_name = source_settings.get("source_name")
source_schema = source_settings.get("source_schema")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse = target_settings.get("target_lakehouse")
target_schema = target_settings.get("target_schema")
target_location = target_settings.get("target_location")
target_load_strategy = target_settings.get("target_load_strategy")

display(table_config)

#### Ingest source table

Read and dedup clean source

In [ ]:
# Read shortcut tables and remove exact duplicates
source_dfs = {}

for cfg in table_config:
    # Get table definition from source_settings
    source_table = cfg.get("source_table")

    # Drop exact duplicates only
    df_source = spark.table(f"{source_schema}.{source_table}")
    df_source = remove_exact_duplicates(df_source)

    # Save source definition on source_dfs
    source_dfs[source_table] = df_source

    #display(source_dfs)
    #print(f"Source preview for {source_schema}.{source_table}")
    #display(df_source.limit(3))

#### Add metadata to source table

Applies metadata and _hashed_pk

In [ ]:
# Add metadata to the cleaned source data
target_dfs = {}

for cfg in table_config:
    # Get table definition from source_settings
    source_table = cfg.get("source_table")
    target_table = cfg.get("target_table")

    # Save source_dfs to target_dfs
    df_target = source_dfs[source_table]

    # Add metadata to target_dfs
    df_target = add_metadata_v2(
        df=df_target,
        source_system=source_name,
        source_table=f"{source_schema}.{source_table}",
        target_table=target_table,
        lineage_id=lineage_id
    )

    # Save new target_dfs with metadata
    target_dfs[target_table] = df_target

    #print(f"Target preview for {target_table}")
    #display(df_target.limit(3))

#### Check duplicates

Return duplicates before merge 

In [ ]:
# Check duplicate business keys before load
for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk")
    
    # Call nb_transformation_shared_functions
    validate_duplicates(target_dfs[target_table], target_pk, 10)

#### Merge to target

Merge to target table based on _hashed_key

In [ ]:
# Load final DataFrames to silver taxonomy tables
load_results = []

for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")

    # Set target table and location
    target_table_name = f"{target_schema}.{target_table}"
    target_location_path = f"{target_location}/{target_table}"
    
    # Lookup target table on target_dfs
    df_target = target_dfs[target_table]

    # Call nb_transformation_shared_functions
    result = load_to_target(df_target, target_table_name, target_load_strategy, target_location_path)

    # Save metrics for pipeline
    load_results.append({
        "target_table": target_table_name,
        "rows_inserted": result["rows_inserted"],
        "rows_updated": result["rows_updated"],
        "rows_read": result["rows_read"],
        "rows_copied": result["rows_copied"]
    })

display(load_results)